In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import torch
assert torch.cuda.is_available(), 'no GPU: switch the runtime to a GPU and rerun'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/temporal-ood-ddi'
os.makedirs(f'{D}/runs/preds', exist_ok=True)
os.makedirs(f'{D}/weights', exist_ok=True)
print(D)

In [ ]:
!mkdir -p /content/w && cp {D}/rev3b_bundle.tgz /content/rev3b_bundle.tgz && tar xzf /content/rev3b_bundle.tgz -C /content/w && cp {D}/inputs.json.gz /content/w/
!pip -q install accelerate sentencepiece protobuf
%cd /content/w

In [ ]:
JOBS = [
 "r3b-marker-sentence-s42",
 "r3b-marker-sentence-s1",
 "r3b-marker-sentence-s2",
 "r3b-marker-sentence-s3",
 "r3b-marker-sentence-s4"
]

In [ ]:
import json, os, sys
sys.path.insert(0, '/content/w')
from ddiood import amend7_panel as pn, gpu_jobs, rev3b_jobs as rj
rj.assert_frozen()
inp = gpu_jobs.Inputs('/content/w/inputs.json.gz')
pn.attach_offsets(inp.test, pn.load_offsets('test'))
texts, fallback = rj.build_inputs(inp)
assert rj.inputs_sha256(texts) == rj.EXPECTED_INPUTS_SHA256, 'the input texts differ from the frozen input list'
print(len(inp.test), 'test pairs', rj.inputs_sha256(texts), fallback)
for c in rj.registry():
    w = f"{D}/weights/{c['weights_name']}"
    assert os.path.isdir(w), f'missing saved weights {w}'
    stored = json.load(open(f"{D}/runs/preds/{c['weights_name']}.json"))
    assert rj.file_sha256(rj.weight_file(w)) == stored['weights']['weight_sha256'], f'weight sha256 differs for {w}'
print('weights present and equal to the stored sha256 for', len(rj.registry()), 'seeds')

In [ ]:
%cd /content/w
!bash -c "set -o pipefail; python -m ddiood.rev3b_jobs run --workdir /content/w --out {D}/runs --weights {D}/weights --only {' '.join(JOBS)} 2>&1 | tee -a {D}/runs/r3b-R3b_marker_sentence.log | (grep --line-buffered -E '^(start|done|skip|FAILED)|Error|Traceback'; true)"

In [ ]:
import os
done = sorted(j for j in JOBS if os.path.exists(f'{D}/runs/preds/{j}.json'))
failed = sorted(j for j in JOBS if os.path.exists(f'{D}/runs/preds/{j}.failed.txt'))
print(f'{len(done)}/{len(JOBS)} jobs finished')
print('missing:', [j for j in JOBS if j not in done])
print('failed:', failed)
for j in failed:
    print(j, open(f'{D}/runs/preds/{j}.failed.txt').read()[-1500:])
assert not failed, failed
assert len(done) == len(JOBS), [j for j in JOBS if j not in done]

In [ ]:
from google.colab import drive
drive.flush_and_unmount()